# TODO

In [ ]:
# Define helper functions for weather (replace with actual data retrieval)
def get_current_temperature(location: str, unit: str) -> float:
    """
    Simulates getting the current temperature at a location.

    Args:
        location: The location to get the temperature for, in the format "City, Country".
        unit: The unit to return the temperature in (e.g., "celsius", "fahrenheit").

    Returns:
        A dummy value (22.0) for demonstration purposes. 
        Replace this with actual temperature retrieval logic.
    """
    return 22.0  # Replace with actual temperature retrieval

def get_current_wind_speed(location: str) -> float:
    """
    Simulates getting the current wind speed at a location.

    Args:
        location: The location to get the wind speed for, in the format "City, Country".

    Returns:
        A dummy value (6.0) for demonstration purposes.
        Replace this with actual wind speed retrieval logic.
    """
    return 6.0  # Replace with actual wind speed retrieval

# Define your tool list (functions the LLM can call)
tools = [get_current_temperature, get_current_wind_speed]

# Assuming you have a pre-loaded tokenizer (replace with your actual loading logic)
# ... tokenizer is loaded ...

# Set up the conversation history
messages = [
  {"role": "system", "content": "You are a bot that responds to weather queries. You should reply with the unit used in the queried location."},
  {"role": "user", "content": "Hey, what's the temperature in Paris right now?"}
]

# Prepare the model input with conversation history, tools, and additional arguments
inputs = tokenizer.apply_chat_template(
    messages,
    chat_template="tool_use",  # Specify the tool calling template
    tools=tools,
    add_generation_prompt=True,  # Add a prompt to guide the LLM
    return_dict=True,
    return_tensors="pt"
)

# Move the inputs to the device the model is on (if using GPU)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate the LLM response with the prepared inputs
out = model.generate(**inputs, max_new_tokens=128)

# Decode the LLM output and print it
print(tokenizer.decode(out[0][len(inputs["input_ids"][0]):]))

# Simulate the LLM suggesting a tool call (based on the conversation)
tool_call_id = "vAHdf3"  # Random and unique ID for each tool call
tool_call = {
  "name": "get_current_temperature",
  "arguments": {"location": "Paris, France", "unit": "celsius"}
}

# Add the suggested tool call to the conversation history
messages.append({
  "role": "assistant",
  "tool_calls": [{"id": tool_call_id, "type": "function", "function": tool_call}]
})

# Simulate executing the tool call (replace with actual function call)
# This retrieves the "dummy" temperature using the defined function
temperature = get_current_temperature(tool_call["arguments"]["location"], tool_call["arguments"]["unit"])

# Add the tool call result to the conversation history
messages.append({
  "role": "tool",
  "tool_call_id": tool_call_id,
  "name": "get_current_temperature",
  "content": str(temperature)  # Convert temperature to string
})

# Prepare the model input again with the updated conversation history
inputs = tokenizer.apply_chat_template(
    messages,
    chat_template="tool_use",
    tools=tools,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

# Move the inputs to the device again
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate the final LLM response with the updated conversation
out = model.generate(**inputs, max_new_tokens=128)

# Decode and print the final LLM response (including temperature)
print(tokenizer.decode(out[0][len(inputs["input_ids"][0]):]))